
# Inspect Whether SB31.5 Satellite Masks Stay Inside the Paper's Central `±300 kpc` Window

这个 notebook 只回答一个问题：

- `props_gals_Fbox_new.pkl` 里，`Fbox-11-eo` 在 `SB31.5` 下的 **全部 satellite 标注**，是不是都落在论文里说的、以宿主星系为中心的 `±300 kpc` 分析区域里。

我们这里 **不看 stream 标注**，只做下面这条主线：

1. 读取原始 `VIS` FITS：`magnitudes-Fbox-11-eo-VIS.fits.gz`
2. 用我们 repo 当前的 `LSBPreprocessor(asinh)` 做 8-bit 渲染
3. 按论文里的尺度换算，画出黄色分析框：
   - 原始 `2051 × 2051` 图上：`937.5 px`
   - legacy `1072 × 1072` 图上：`490.0048756704 px`
4. 从 `props_gals_Fbox_new.pkl` 里读取 `SB31.5` 的全部 satellite masks
5. 把这些 masks 全部画在图上，直接看它们是不是都在黄色框里面


In [ ]:

from pathlib import Path
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.io import load_fits_gz, SatelliteDataLoader
from src.data.preprocessing import LSBPreprocessor

plt.rcParams['figure.dpi'] = 140
plt.rcParams['figure.figsize'] = (14, 8)


def first_existing(candidates):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError('No candidate path exists:\n' + '\n'.join(str(p) for p in candidates))


GALAXY_ID = 11
ORIENTATION = 'eo'
SAT_SB_THRESHOLD = 31.5
PIXEL_SCALE_KPC = 0.64
ANALYSIS_HALF_WIDTH_KPC = 300.0
LEGACY_SIZE = (1072, 1072)   # (width, height)
MODERN_SIZE = (1024, 1024)   # (width, height)
NONLINEARITY = 200.0
ZEROPOINT = 22.5
CLIP_PERCENTILE = 99.5

RAW_FITS = first_existing([
    PROJECT_ROOT / 'data/01_raw/LSB_and_Satellites/fbox/sb_maps/magnitudes-Fbox-11-eo-VIS.fits.gz',
    Path('/home/yuqyan/Yuqi/LSB-AI-Detection/data/01_raw/LSB_and_Satellites/fbox/sb_maps/magnitudes-Fbox-11-eo-VIS.fits.gz'),
])

PKL_PATH = first_existing([
    Path('/shares/feldmann.ics.mnf.uzh/Lucas/pNbody/satellites/fbox/props_gals_Fbox_new.pkl'),
])

print('RAW_FITS =', RAW_FITS)
print('PKL_PATH =', PKL_PATH)



## 1. Helper Functions


In [ ]:

def image_to_rgb(arr):
    if arr.ndim == 2:
        return np.stack([arr] * 3, axis=-1)
    if arr.ndim == 3 and arr.shape[2] == 4:
        return arr[:, :, :3]
    return arr


def resize_mask_nearest(mask_bool, size_wh):
    return cv2.resize(mask_bool.astype(np.uint8), size_wh, interpolation=cv2.INTER_NEAREST).astype(bool)


def centered_window_box(shape_hw, full_size_px):
    h, w = shape_hw
    cx = (w - 1) / 2.0
    cy = (h - 1) / 2.0
    half = full_size_px / 2.0
    return (cx - half, cy - half, cx + half, cy + half)


def make_window_mask(shape_hw, window_box_xyxy):
    h, w = shape_hw
    x0, y0, x1, y1 = window_box_xyxy
    xs = np.arange(w)
    ys = np.arange(h)
    xmask = (xs >= x0) & (xs < x1)
    ymask = (ys >= y0) & (ys < y1)
    return np.outer(ymask, xmask)


def add_window_box(ax, box_xyxy, color='yellow', linewidth=2.5, label=None):
    x0, y0, x1, y1 = box_xyxy
    rect = Rectangle((x0, y0), x1 - x0, y1 - y0,
                     fill=False, edgecolor=color, linewidth=linewidth)
    ax.add_patch(rect)
    if label:
        ax.text(x0, max(0, y0 - 12), label, color=color, fontsize=10,
                bbox=dict(facecolor='black', alpha=0.65, pad=2, edgecolor='none'))


def overlay_union_mask(image_rgb, mask_bool, color=(255, 80, 80), alpha=0.35):
    image_rgb = image_to_rgb(image_rgb)
    out = image_rgb.astype(np.float32).copy()
    color = np.array(color, dtype=np.float32)
    out[mask_bool] = (1.0 - alpha) * out[mask_bool] + alpha * color
    return np.clip(out, 0, 255).astype(np.uint8)


def draw_mask_contours(ax, masks, cmap_name='tab20', linewidth=1.2):
    if not masks:
        return
    cmap = plt.get_cmap(cmap_name, max(len(masks), 1))
    for i, mask in enumerate(masks):
        color = cmap(i)
        ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=linewidth)


def fraction_inside_window(mask_bool, window_mask_bool):
    total = int(mask_bool.sum())
    inside = int(np.logical_and(mask_bool, window_mask_bool).sum())
    outside = int(np.logical_and(mask_bool, ~window_mask_bool).sum())
    return {
        'total_px': total,
        'inside_px': inside,
        'outside_px': outside,
        'inside_fraction': (inside / total) if total else float('nan'),
        'outside_fraction': (outside / total) if total else float('nan'),
    }



## 2. Render the Raw `VIS` FITS with the Repo's Own Asinh 8-bit Pipeline, Then Project the Paper's `±300 kpc` Window

这里不去复刻旧 legacy log transform，而是按你的要求，直接走我们当前 repo 的 `LSBPreprocessor(asinh)`：

- 原始输入：`magnitudes-Fbox-11-eo-VIS.fits.gz`
- 参数：`zeropoint=22.5`, `nonlinearity=200`, `clip_percentile=99.5`
- 输出：保留原始 `2051 × 2051`，并额外生成 `1072 × 1072` 和 `1024 × 1024`

同时，我们会把论文中的尺度关系直接换算成像素：

- 原始图上：`300 kpc / 0.64 kpc per pixel = 468.75 px`，也就是一个 `937.5 px` 边长的中心窗口。
- legacy `1072` 图上：这个窗口应该变成 `937.5 × 1072 / 2051 = 490.0048756704 px`。


In [ ]:

sb_map = load_fits_gz(RAW_FITS)
raw_h, raw_w = sb_map.shape

asinh_full = LSBPreprocessor(
    zeropoint=ZEROPOINT,
    nonlinearity=NONLINEARITY,
    clip_percentile=CLIP_PERCENTILE,
    target_size=(raw_w, raw_h),
).process(sb_map)

asinh_1072 = LSBPreprocessor(
    zeropoint=ZEROPOINT,
    nonlinearity=NONLINEARITY,
    clip_percentile=CLIP_PERCENTILE,
    target_size=LEGACY_SIZE,
).process(sb_map)

asinh_1024 = LSBPreprocessor(
    zeropoint=ZEROPOINT,
    nonlinearity=NONLINEARITY,
    clip_percentile=CLIP_PERCENTILE,
    target_size=MODERN_SIZE,
).process(sb_map)

analysis_half_px_2051 = ANALYSIS_HALF_WIDTH_KPC / PIXEL_SCALE_KPC
analysis_full_px_2051 = 2.0 * analysis_half_px_2051
analysis_full_px_1072 = analysis_full_px_2051 * LEGACY_SIZE[0] / raw_w
analysis_full_px_1024 = analysis_full_px_2051 * MODERN_SIZE[0] / raw_w

analysis_box_2051 = centered_window_box((raw_h, raw_w), analysis_full_px_2051)
analysis_box_1072 = centered_window_box((LEGACY_SIZE[1], LEGACY_SIZE[0]), analysis_full_px_1072)
analysis_box_1024 = centered_window_box((MODERN_SIZE[1], MODERN_SIZE[0]), analysis_full_px_1024)

analysis_window_2051 = make_window_mask((raw_h, raw_w), analysis_box_2051)
analysis_window_1072 = make_window_mask((LEGACY_SIZE[1], LEGACY_SIZE[0]), analysis_box_1072)
analysis_window_1024 = make_window_mask((MODERN_SIZE[1], MODERN_SIZE[0]), analysis_box_1024)

print('Raw FITS shape:', sb_map.shape)
print('Asinh full render shape :', asinh_full.shape)
print('Asinh 1072 render shape :', asinh_1072.shape)
print('Asinh 1024 render shape :', asinh_1024.shape)
print()
print(f'analysis_half_px_2051 = {analysis_half_px_2051}')
print(f'analysis_full_px_2051 = {analysis_full_px_2051}')
print(f'analysis_full_px_1072 = {analysis_full_px_1072}')
print(f'analysis_full_px_1024 = {analysis_full_px_1024}')


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(asinh_full)
axes[0].set_title('Repo asinh render at original 2051x2051')
add_window_box(axes[0], analysis_box_2051, color='yellow', label='paper ±300 kpc window (~937.5 px)')
axes[0].axis('off')

axes[1].imshow(asinh_1072)
axes[1].set_title('Repo asinh render resized to 1072x1072')
add_window_box(axes[1], analysis_box_1072, color='yellow', label='resized window (~490.0049 px)')
axes[1].axis('off')

axes[2].imshow(asinh_1024)
axes[2].set_title('Repo asinh render resized to 1024x1024')
add_window_box(axes[2], analysis_box_1024, color='yellow', label='same window after resize')
axes[2].axis('off')

plt.tight_layout()
plt.show()



## 3. Load All `SB31.5` Satellite Masks from the PKL and Draw Them on the Images

这里我们只看 satellite：

- 从 `props_gals_Fbox_new.pkl` 读取 `Fbox-11-eo` 在 `SB31.5` 下的全部 satellite 标注
- 不看 stream
- 把 **所有 satellite masks** 都画在图上
- 你只需要看：这些 mask 有没有跑到黄色框外面去


In [ ]:

loader = SatelliteDataLoader(PKL_PATH, lazy_load=True)
satellites = loader.get_satellites(GALAXY_ID, ORIENTATION, SAT_SB_THRESHOLD, load_seg_map=False)

sat_masks_2051 = [sat.get_binary_mask(shape=sb_map.shape).astype(bool) for sat in satellites.satellites]
sat_masks_2051 = [m for m in sat_masks_2051 if m.any()]

sat_masks_1072 = [resize_mask_nearest(m, LEGACY_SIZE) for m in sat_masks_2051]
sat_masks_1024 = [resize_mask_nearest(m, MODERN_SIZE) for m in sat_masks_2051]

sat_union_2051 = np.logical_or.reduce(sat_masks_2051) if sat_masks_2051 else np.zeros(sb_map.shape, dtype=bool)
sat_union_1072 = np.logical_or.reduce(sat_masks_1072) if sat_masks_1072 else np.zeros((LEGACY_SIZE[1], LEGACY_SIZE[0]), dtype=bool)
sat_union_1024 = np.logical_or.reduce(sat_masks_1024) if sat_masks_1024 else np.zeros((MODERN_SIZE[1], MODERN_SIZE[0]), dtype=bool)

union_stats_2051 = fraction_inside_window(sat_union_2051, analysis_window_2051)
union_stats_1072 = fraction_inside_window(sat_union_1072, analysis_window_1072)
union_stats_1024 = fraction_inside_window(sat_union_1024, analysis_window_1024)

per_mask_inside_2051 = [bool(np.logical_and(mask, ~analysis_window_2051).sum() == 0) for mask in sat_masks_2051]
per_mask_inside_1072 = [bool(np.logical_and(mask, ~analysis_window_1072).sum() == 0) for mask in sat_masks_1072]
per_mask_inside_1024 = [bool(np.logical_and(mask, ~analysis_window_1024).sum() == 0) for mask in sat_masks_1024]

print('SB threshold:', SAT_SB_THRESHOLD)
print('Number of satellite instances:', len(sat_masks_2051))
print('Satellite ids:', [sat.id for sat in satellites.satellites])
print()
print('Union mask stats in 2051 space:')
print(union_stats_2051)
print('Union mask stats in 1072 space:')
print(union_stats_1072)
print('Union mask stats in 1024 space:')
print(union_stats_1024)
print()
print('How many individual masks are fully inside the yellow box?')
print(f'  2051 space : {sum(per_mask_inside_2051)} / {len(per_mask_inside_2051)}')
print(f'  1072 space : {sum(per_mask_inside_1072)} / {len(per_mask_inside_1072)}')
print(f'  1024 space : {sum(per_mask_inside_1024)} / {len(per_mask_inside_1024)}')


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

img0 = overlay_union_mask(asinh_full, sat_union_2051, color=(255, 80, 80), alpha=0.25)
axes[0].imshow(img0)
draw_mask_contours(axes[0], sat_masks_2051, cmap_name='tab20', linewidth=1.0)
add_window_box(axes[0], analysis_box_2051, color='yellow', label='paper ±300 kpc window')
axes[0].set_title(f'2051 render + all SB31.5 satellite masks\ninside box: {sum(per_mask_inside_2051)}/{len(per_mask_inside_2051)}')
axes[0].axis('off')

img1 = overlay_union_mask(asinh_1072, sat_union_1072, color=(255, 80, 80), alpha=0.25)
axes[1].imshow(img1)
draw_mask_contours(axes[1], sat_masks_1072, cmap_name='tab20', linewidth=1.0)
add_window_box(axes[1], analysis_box_1072, color='yellow', label='expected 490 px window')
axes[1].set_title(f'1072 render + all SB31.5 satellite masks\ninside box: {sum(per_mask_inside_1072)}/{len(per_mask_inside_1072)}')
axes[1].axis('off')

img2 = overlay_union_mask(asinh_1024, sat_union_1024, color=(255, 80, 80), alpha=0.25)
axes[2].imshow(img2)
draw_mask_contours(axes[2], sat_masks_1024, cmap_name='tab20', linewidth=1.0)
add_window_box(axes[2], analysis_box_1024, color='yellow', label='same window after resize')
axes[2].set_title(f'1024 render + all SB31.5 satellite masks\ninside box: {sum(per_mask_inside_1024)}/{len(per_mask_inside_1024)}')
axes[2].axis('off')

plt.tight_layout()
plt.show()



## 4. What You Should Look At

你现在只需要看上一格：

- 黄色框 = 论文里 `±300 kpc` 的理论分析窗口
- 彩色轮廓 = `SB31.5` 下的全部 satellite masks
- 标题里的 `inside box: x / N` = 有多少个单独的 satellite mask 完全落在黄色框里

如果你看到：

- 大部分甚至全部 masks 都在黄色框里，那你的怀疑就是有支持的。
- 明显有不少 masks 跑到黄色框外面，那就说明“标注只覆盖中心 `±300 kpc`”这个想法至少不能直接成立。
